# Laboratorio 7 — Spark MLlib
## 01. Carga, armonización y calidad de datos

Este notebook prepara la población analítica de salarios de personas asalariadas con las bases de **Personas** de la ENEIC:

- **2025 (I a IV)**: desarrollo y entrenamiento.
- **2026 (I)**: prueba final.

Pasos: identificar la procedencia de cada archivo, seleccionar y homologar columnas, unir los cuatro archivos de 2025 con `unionByName`, medir faltantes, aplicar los filtros en un orden fijo, verificar la unicidad de las claves y guardar los conjuntos preparados en Parquet.

Las bases se leen con pandas/openpyxl **una por una**, se convierten a Spark con tipos explícitos y se guardan en Parquet. Desde ahí todo el procesamiento es con Spark.

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ))

import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from src import config, carga, calidad

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

spark = config.crear_spark("lab7_01_calidad")
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

## 1. Procedencia y encabezados de los archivos

Antes de cargar se revisan los encabezados de cada xlsx. Esto sirve para documentar el número de columnas originales y para comprobar si el orden de las columnas es el mismo entre archivos.

In [ ]:
encabezados = {p: carga.columnas_del_archivo(carga.ruta_archivo(p)) for p in config.PERIODOS}

resumen_cols = pd.DataFrame({
    "periodo_archivo": list(encabezados),
    "archivo": [config.ARCHIVOS[p] for p in encabezados],
    "columnas_originales": [len(c) for c in encabezados.values()],
})
base = encabezados["2025T1"]
resumen_cols["columnas_en_distinta_posicion_vs_2025T1"] = [
    sum(1 for i, c in enumerate(cols) if i >= len(base) or base[i] != c) for cols in encabezados.values()
]
resumen_cols["columnas_requeridas_presentes"] = [
    all(c in cols for c in config.COLUMNAS_ORIGINALES) for cols in encabezados.values()
]
resumen_cols

In [ ]:
extra_2025t4 = [c for c in encabezados["2025T4"] if c not in set(encabezados["2025T3"])]
print(f"Columnas de 2025T4 que no existen en 2025T3: {len(extra_2025t4)}")
print(extra_2025t4[:20], "..." if len(extra_2025t4) > 20 else "")

## 2. Carga, homologación de tipos y procedencia

Cada archivo se convierte una sola vez a Parquet tipado en `data/processed/tipado/`. Se seleccionan únicamente las 14 columnas requeridas y se les da un tipo explícito:

- **Numéricas** (`salario_mensual`, `edad`, `antiguedad_anios`, `antiguedad_meses`, `horas_semanales`, `FACTOR`): `double`.
- **Enteras** (`ocupado`, `ANIO`, `TRIMESTRE`): `int`.
- **Códigos** (`NUM_HOGAR`, `NUM_PERSONA`, `nivel_educativo`, `categoria_ocupacional`, `dominio`): texto canónico, de modo que `1`, `1.0` y ` 1 ` se representan igual.

Se agregan `periodo_archivo`, `anio_archivo`, `trimestre_calendario` y `archivo_origen` a partir del archivo de procedencia. `TRIMESTRE` se conserva tal como llegó y **no se usa** como trimestre calendario.

In [ ]:
dfs = carga.cargar_periodos(spark, list(config.PERIODOS))
for p, d in dfs.items():
    d.persist()
    print(p, d.count(), "registros")

### Verificación de `TRIMESTRE` frente al archivo de procedencia

Se confirma que el valor original de `TRIMESTRE` no coincide con el trimestre calendario y que un mismo archivo puede traer más de un valor.

In [ ]:
todo = carga.unir_periodos(list(dfs.values()))
(todo.groupBy("periodo_archivo", "trimestre_calendario", "TRIMESTRE").count()
     .orderBy("periodo_archivo", "TRIMESTRE").toPandas())

## 3. Unión de los cuatro archivos de 2025 con `unionByName`

Se apilan por **nombre** de columna. Como cada DataFrame ya contiene únicamente las columnas seleccionadas, la unión no depende de las 302 columnas de IV de 2025 ni de su orden.

In [ ]:
df_2025 = carga.unir_periodos([dfs[p] for p in config.PERIODOS_TRAIN]).persist()
df_2026 = dfs[config.PERIODO_TEST]

print("Columnas 2025:", len(df_2025.columns), "| Columnas 2026:", len(df_2026.columns))
print("Mismas columnas y orden:", df_2025.columns == df_2026.columns)
df_2025.printSchema()

In [ ]:
df_2025.select(
    "periodo_archivo", "anio_archivo", "trimestre_calendario", "archivo_origen",
    "salario_mensual", "edad", "antiguedad_anios", "antiguedad_meses", "horas_semanales",
    "nivel_educativo", "categoria_ocupacional", "dominio", "ocupado",
    "NUM_HOGAR", "NUM_PERSONA", "FACTOR", "ANIO", "TRIMESTRE",
).limit(5).toPandas()

## 4. Faltantes por variable, antes de aplicar los filtros

Se cuentan como faltantes los valores nulos o `NaN` de cada variable seleccionada, sobre todos los registros originales de cada conjunto.

In [ ]:
variables = [c for c in df_2025.columns if c not in ("periodo_archivo", "anio_archivo", "trimestre_calendario", "archivo_origen")]

falt_2025 = calidad.faltantes(df_2025, variables)
falt_2026 = calidad.faltantes(df_2026, variables)
falt = falt_2025.merge(falt_2026, on="variable", suffixes=("_2025", "_2026"))
falt[["variable", "faltantes_2025", "porcentaje_2025", "faltantes_2026", "porcentaje_2026"]]

In [ ]:
por_periodo = pd.concat([calidad.faltantes(d, ["salario_mensual", "categoria_ocupacional", "antiguedad_anios", "horas_semanales"]).assign(periodo_archivo=p)
                         for p, d in dfs.items()])
por_periodo.pivot(index="variable", columns="periodo_archivo", values="porcentaje")

### Interpretación de los faltantes

Los faltantes no significan lo mismo en todas las variables. Variables como el salario, la categoría ocupacional, la antigüedad o las horas habituales corresponden a preguntas del módulo de empleo, que solo se formulan a personas ocupadas: para el resto de la población el campo queda vacío porque la pregunta **no corresponde**, no porque falte la respuesta. Por eso los porcentajes altos en esas variables son esperables antes de filtrar.

En cambio, un faltante en una variable de una persona que sí debía responder (por ejemplo, una persona ocupada asalariada sin salario) es una **respuesta no registrada**, y ese sí es un problema de calidad. El embudo de la sección 6 los separa: primero se restringe a quienes cumplen las condiciones de la población analítica y luego se cuentan las exclusiones por dato inválido o ausente. El salario no se imputa.

## 5. Validación de códigos categóricos contra el diccionario

Se revisan los códigos observados de las variables categóricas. Los códigos ausentes o no reconocidos se representarán como `DESCONOCIDO`; el código educativo `0` significa "ninguno" y **no** es un faltante.

In [ ]:
for col in config.CATEGORICAS:
    print(f"--- {col} (2025 y 2026 juntos)")
    tabla = calidad.codigos_observados(df_2025.unionByName(df_2026), col)
    print(tabla.to_string(index=False))
    print("Códigos no reconocidos:", int(tabla.loc[~tabla["reconocido"], "count"].sum()), "registros\n")

## 6. Filtros en orden fijo y registros excluidos

Los filtros se aplican siempre en el mismo orden y cada uno se cuenta sobre los registros que llegaron hasta él:

1. Edad finita y mayor o igual a 15.
2. Ocupado (`OCUPADOS = 1`).
3. Asalariado (`P05C16` en 1, 2, 3 o 4).
4. Salario (`P05D01`) numérico, finito y estrictamente positivo.
5. Antigüedad válida: años no negativos, meses enteros entre 0 y 11, antigüedad calculada (`años + meses / 12`) no negativa y menor o igual a la edad.
6. Horas habituales mayores que cero y menores o iguales a 168 por semana.

Un registro que no permite evaluar un criterio (valor ausente) se excluye y se contabiliza en ese paso. Los salarios altos o bajos **no** se eliminan por ser extremos.

In [ ]:
embudo_2025 = calidad.embudo(df_2025)
embudo_2026 = calidad.embudo(df_2026)

def formato(tabla):
    return tabla.pivot(index="paso", columns="periodo_archivo", values="excluidos")

pasos_orden = ["registros originales"] + [n for n, _ in calidad.pasos_filtro()]
print("Registros que sobreviven tras cada paso (2025)")
display(embudo_2025.pivot(index="paso", columns="periodo_archivo", values="restantes").reindex(pasos_orden))
print("Registros excluidos en cada paso (2025)")
display(formato(embudo_2025).reindex(pasos_orden))

In [ ]:
print("2026T1")
display(embudo_2026.set_index("paso").loc[pasos_orden, ["restantes", "excluidos"]])

### Registros por archivo antes y después de los filtros

In [ ]:
por_archivo = pd.concat([calidad.resumen_por_archivo(embudo_2025), calidad.resumen_por_archivo(embudo_2026)])
por_archivo = por_archivo[por_archivo["periodo_archivo"] != "TOTAL"].reset_index(drop=True)
display(por_archivo)

ax = por_archivo.set_index("periodo_archivo")[["antes", "despues"]].plot.bar(figsize=(8, 4), rot=0)
ax.set_ylabel("registros")
ax.set_title("Registros por archivo antes y después de los filtros")
plt.tight_layout()
plt.savefig(config.FIGURES / "01_registros_por_archivo.png", dpi=120)
plt.show()

In [ ]:
total = embudo_2025[embudo_2025["periodo_archivo"] == "TOTAL"].set_index("paso")
inicial = int(total.loc["registros originales", "restantes"])
final = int(total["restantes"].iloc[-1])
mayor = total.drop("registros originales")["excluidos"].idxmax()
print(f"2025: {inicial:,} registros originales -> {final:,} en la población analítica ({100 * final / inicial:.1f} %).")
print(f"El paso que más registros excluye es: '{mayor}' ({int(total.loc[mayor, 'excluidos']):,}).")

### Interpretación del embudo

Los primeros pasos (edad, ocupación y categoría asalariada) definen la población analítica, porque la base cubre a toda la población y solo una parte está ocupada y asalariada. Esas exclusiones son **cambios de población**, no errores de datos. Las exclusiones de los pasos de salario, antigüedad y horas sí reflejan datos ausentes o inconsistentes entre personas que cumplían las condiciones anteriores. Como el orden es fijo, cada registro se cuenta una sola vez, en el primer criterio que no cumple.

Al exigir un salario positivo registrado, los resultados se refieren específicamente a **asalariados con salario positivo registrado**.

## 7. Unicidad de `periodo_archivo`, `NUM_HOGAR` y `NUM_PERSONA`

Se comprueba, sobre todos los registros originales seleccionados, si la combinación es única. Si hay claves repetidas se investiga si son **repeticiones exactas** (todas las columnas seleccionadas iguales) o **registros en conflicto** (mismos identificadores con valores distintos). No se usa `dropDuplicates()`.

In [ ]:
unicidad = pd.DataFrame({
    "2025": calidad.resumen_unicidad(df_2025),
    "2026": calidad.resumen_unicidad(df_2026),
})
unicidad

In [ ]:
detalle_2025 = calidad.detalle_duplicados(df_2025)
detalle_2026 = calidad.detalle_duplicados(df_2026)
tipos = (detalle_2025.withColumn("conjunto", F.lit("2025"))
         .unionByName(detalle_2026.withColumn("conjunto", F.lit("2026")))
         .groupBy("conjunto", "tipo").agg(F.count(F.lit(1)).alias("claves"), F.sum("filas").alias("filas")))
tipos.orderBy("conjunto", "tipo").toPandas()

In [ ]:
n_dup = unicidad.loc["claves_duplicadas", "2025"]
if n_dup == 0:
    print("2025: no hay claves repetidas; cada (periodo_archivo, NUM_HOGAR, NUM_PERSONA) aparece una sola vez.")
else:
    print(f"2025: {n_dup:,} claves repetidas. Columnas que difieren dentro de esas claves:")
    display(calidad.columnas_en_conflicto(df_2025).query("claves_con_diferencia > 0"))
    print("Ejemplos de claves repetidas:")
    display(detalle_2025.orderBy(F.desc("versiones"), F.desc("filas")).limit(10).toPandas())

### Interpretación de la unicidad

Una clave repetida **exacta** indica que el mismo registro llegó más de una vez; una **en conflicto** indica que hay dos versiones distintas de la misma persona en el mismo corte y no se puede decidir cuál es la correcta sin consultar la fuente. Por eso los duplicados no se ocultan con `dropDuplicates()`: primero se cuantifican y se entiende su origen. Los conteos anteriores quedan como evidencia y, si existieran claves repetidas dentro de la población analítica, se reportan como una limitación.

Como `periodo_archivo` forma parte de la clave, una misma persona observada en dos periodos **no** se considera repetida (ver pregunta 3).

## 8. Conjuntos preparados en Parquet

Se aplican los filtros de la sección 6, se validan las categóricas (`DESCONOCIDO` para ausentes o no reconocidos) y se guardan por separado la población de 2025 y la de 2026. La antigüedad queda en años. Para 2026 se aplican **exactamente las mismas reglas** que para 2025.

In [ ]:
prep_2025 = calidad.preparar(df_2025)
prep_2026 = calidad.preparar(df_2026)

ruta_2025 = calidad.guardar_preparado(prep_2025, "personas_2025")
ruta_2026 = calidad.guardar_preparado(prep_2026, "personas_2026")

leido_2025 = spark.read.parquet(str(ruta_2025))
leido_2026 = spark.read.parquet(str(ruta_2026))
print("personas_2025:", leido_2025.count(), "registros |", ruta_2025.name)
print("personas_2026:", leido_2026.count(), "registros |", ruta_2026.name)
leido_2025.printSchema()

In [ ]:
leido_2025.limit(5).toPandas()

In [ ]:
desconocidos = pd.DataFrame({
    conjunto: {c: leido.filter(F.col(c) == config.DESCONOCIDO).count() for c in config.CATEGORICAS}
    for conjunto, leido in {"2025": leido_2025, "2026": leido_2026}.items()
})
desconocidos.index.name = "registros con DESCONOCIDO"
desconocidos

## 9. Preguntas de la sección

**1. ¿Por qué IV de 2025 no puede apilarse por posición de columnas con los otros archivos?**

Porque tiene una estructura distinta: 302 columnas frente a las 270 de los demás archivos, y las columnas adicionales alteran las posiciones (la tabla de la sección 1 muestra cuántas columnas quedan en otra posición). Apilar por posición pondría en una misma columna variables con significados diferentes, por ejemplo edad con horas o el salario con un código, sin error visible, y con tipos mezclados. `unionByName` empareja cada columna por su nombre, así que solo se combinan variables equivalentes y el orden o las columnas sobrantes dejan de importar.

**2. ¿Qué diferencia existe entre un dato ausente porque la pregunta no corresponde y una respuesta no registrada?**

El primero es un faltante estructural: la pregunta no se le hace a esa persona por el flujo del cuestionario (por ejemplo, el salario de la ocupación principal no se pregunta a quien no trabaja). No es un error y no debe imputarse ni tratarse como pérdida de información. El segundo ocurre cuando la persona sí debía responder y el dato no quedó registrado (no respondió, no supo o hubo un error de captura); ese sí reduce la calidad del dato. Se distinguen mirando el flujo del cuestionario y la población a la que aplica la pregunta: por eso primero se define la población analítica y solo entonces se cuentan los faltantes que quedan. En este trabajo el salario ausente entre asalariados se excluye y no se imputa, y las categóricas ausentes o no reconocidas se marcan como `DESCONOCIDO`, sin convertirlas a cero (el código educativo 0 significa "ninguno").

**3. ¿Por qué una persona observada en dos períodos no debe eliminarse como duplicado del conjunto longitudinal?**

Porque la encuesta tiene un diseño longitudinal con rotación: la misma persona puede ser entrevistada en más de un trimestre y cada aparición es una observación válida de un período distinto. Por eso la clave de unicidad incluye `periodo_archivo`. Eliminarlas descartaría información legítima, cambiaría la composición de cada trimestre y alteraría las comparaciones entre períodos. Quedan, eso sí, dos consecuencias que se deben recordar: el número de filas no es el número de personas distintas y las observaciones repetidas de una misma persona no son independientes entre sí.

**4. ¿Por qué el número de registros de la base filtrada no representa a todos los trabajadores del país?**

Porque es una muestra, no un censo: cada registro representa a un número distinto de personas según su `FACTOR` de expansión, y aquí los análisis son no ponderados. Además, la base filtrada solo incluye personas de 15 años o más, ocupadas, asalariadas y con salario positivo registrado, con lo que quedan fuera los trabajadores independientes, los patronos, los no remunerados y quienes no tienen salario registrado. Y hay personas que aparecen en más de un período. Los resultados describen los registros analizados; para estimar a la población se ponderaría con `FACTOR` y se consideraría el diseño muestral.

In [ ]:
for d in list(dfs.values()) + [df_2025]:
    d.unpersist()
spark.stop()